In [ ]:

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

FOUNDRY_DIR = Path("/content/drive/MyDrive/foundry")
DATA_DIR = FOUNDRY_DIR / "tiv-mt560-v1"

SMOKE_RUN_DIR = FOUNDRY_DIR / "runs/tiv-lora-smoke-v1"
ADAPTER_DIR = SMOKE_RUN_DIR / "best_adapter"

EVALUATION_DIR = (
    FOUNDRY_DIR / "evaluations/tiv-lora-smoke-v1"
)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

assert (DATA_DIR / "validation.jsonl").exists()
assert (DATA_DIR / "test.jsonl").exists()
assert (SMOKE_RUN_DIR / "training_config.json").exists()
assert (SMOKE_RUN_DIR / "training_metrics.json").exists()
assert (ADAPTER_DIR / "adapter_config.json").exists()

print("Dataset:", DATA_DIR)
print("Adapter:", ADAPTER_DIR)
print("Evaluation output:", EVALUATION_DIR)
print("Artifacts verified.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset: /content/drive/MyDrive/foundry/tiv-mt560-v1
Adapter: /content/drive/MyDrive/foundry/runs/tiv-lora-smoke-v1/best_adapter
Evaluation output: /content/drive/MyDrive/foundry/evaluations/tiv-lora-smoke-v1
Artifacts verified.


In [ ]:
# Locate repository and Install Dependencies


from pathlib import Path
import os
import subprocess
import sys

FOUNDRY_DIR = Path(
    "/content/drive/MyDrive/foundry/YiM-N-ATLAS-Language-Foundry"
)

assert FOUNDRY_DIR.exists(), f"Repository not found: {FOUNDRY_DIR}"

os.chdir(FOUNDRY_DIR)

print("Repository:", Path.cwd())
print("Python:", sys.version)
print("Branch / commit:")

subprocess.run(
    ["git", "log", "-1", "--oneline"],
    check=False,
)

print("\nRelevant project files:")
for relative_path in [
    "foundry/training/preprocessing.py",
    "foundry/training/model_setup.py",
    "foundry/training/collator.py",
    "api/schemas/experiment.py",
]:
    path = FOUNDRY_DIR / relative_path
    print(f"{'FOUND' if path.exists() else 'MISSING'}  {relative_path}")


Repository: /content/drive/MyDrive/foundry/YiM-N-ATLAS-Language-Foundry
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Branch / commit:

Relevant project files:
FOUND  foundry/training/preprocessing.py
FOUND  foundry/training/model_setup.py
FOUND  foundry/training/collator.py
FOUND  api/schemas/experiment.py


In [ ]:
# Inspect Actual Training prompt

from pathlib import Path

preprocessing_file = FOUNDRY_DIR / "foundry/training/preprocessing.py"

print(preprocessing_file.read_text())


from dataclasses import dataclass
from foundry.datasets.record import DatasetRecord

IGNORE_INDEX = -100


@dataclass(frozen=True)
class TokenizedTrainingExample:
    record_id: str
    language_code: str
    input_ids: list[int]
    attention_mask: list[int]
    labels: list[int]


class NAtlasPreprocessor:
    """
    Tokenizes canonical Foundry DatasetRecords using the N-ATLaS chat template,
    masks the prompt portion with IGNORE_INDEX (-100), and retains supervision
    only on the assistant response tokens (including end-of-turn tokens).
    """

    def __init__(self, tokenizer, max_length: int = 512):
        if max_length <= 0:
            raise ValueError("max_length must be positive")
        self.tokenizer = tokenizer
        self.max_length = max_length

    def preprocess(
        self, record: DatasetRecord
    ) -> TokenizedTrainingExample:
        if not record.input.strip():
            raise ValueError("input cannot be empty")
        if not record.target.strip():
 

In [ ]:
# Inspect the saved training artifacts

import json

with open(SMOKE_RUN_DIR / "training_config.json", encoding="utf-8") as f:
    training_config = json.load(f)

with open(SMOKE_RUN_DIR / "training_metrics.json", encoding="utf-8") as f:
    training_metrics = json.load(f)

print("Training configuration:")
print(json.dumps(training_config, indent=2))

print("\nSmoke-training metrics:")
print(json.dumps(training_metrics, indent=2))

print("\nAdapter files:")
for path in sorted(ADAPTER_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size:,} bytes")


Training configuration:
{
  "per_device_batch_size": 1,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0001,
  "epochs": 1,
  "warmup_ratio": 0.0,
  "weight_decay": 0.01,
  "seed": 42
}

Smoke-training metrics:
{
  "epochs_completed": 1,
  "optimizer_steps": 4,
  "total_optimizer_steps": 4,
  "warmup_steps": 0,
  "best_validation_loss": 6.787520707663843,
  "history": [
    {
      "epoch": 1,
      "training_loss": 5.756267112308584,
      "validation_loss": 6.787520707663843,
      "optimizer_steps": 4,
      "learning_rate": 0.0001
    }
  ]
}

Adapter files:
README.md: 5,182 bytes
adapter_config.json: 1,093 bytes
adapter_model.safetensors: 27,280,152 bytes


In [ ]:
#Select a reproducible validation sample

import json
import random

VALIDATION_FILE = DATA_DIR / "validation.jsonl"
EVALUATION_SAMPLE_FILE = EVALUATION_DIR / "validation_sample.jsonl"

with open(VALIDATION_FILE, encoding="utf-8") as f:
    validation_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert validation_records, "Validation dataset is empty."

sample_size = min(10, len(validation_records))
rng = random.Random(42)

evaluation_records = rng.sample(validation_records, sample_size)

with open(EVALUATION_SAMPLE_FILE, "w", encoding="utf-8") as f:
    for record in evaluation_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Validation records available: {len(validation_records):,}")
print(f"Evaluation sample size: {len(evaluation_records)}")
print(f"Saved sample: {EVALUATION_SAMPLE_FILE}")

print("\nFirst example — inspect the fields, not just the translation:")
print(json.dumps(evaluation_records[0], ensure_ascii=False, indent=2))


Validation records available: 18,410
Evaluation sample size: 10
Saved sample: /content/drive/MyDrive/foundry/evaluations/tiv-lora-smoke-v1/validation_sample.jsonl

First example — inspect the fields, not just the translation:
{
  "id": "mt560-140929",
  "input": "This advance knowledge brought with it responsibility .",
  "target": "Er Yehova a pasen mbatomov nav kwagh u ken hemen yô , kwagh ne gema va ve a tom .",
  "language_code": "tiv"
}


In [ ]:

import inspect
from foundry.training.preprocessing import NAtlasPreprocessor

print(inspect.getsource(NAtlasPreprocessor))


class NAtlasPreprocessor:
    """
    Tokenizes canonical Foundry DatasetRecords using the N-ATLaS chat template,
    masks the prompt portion with IGNORE_INDEX (-100), and retains supervision
    only on the assistant response tokens (including end-of-turn tokens).
    """

    def __init__(self, tokenizer, max_length: int = 512):
        if max_length <= 0:
            raise ValueError("max_length must be positive")
        self.tokenizer = tokenizer
        self.max_length = max_length

    def preprocess(
        self, record: DatasetRecord
    ) -> TokenizedTrainingExample:
        if not record.input.strip():
            raise ValueError("input cannot be empty")
        if not record.target.strip():
            raise ValueError("target cannot be empty")

        messages = [
            {"role": "user", "content": record.input},
            {"role": "assistant", "content": record.target},
        ]

        prompt_result = self.tokenizer.apply_chat_template(
            messages[

In [ ]:

%pip install -q "transformers==5.16.1" "peft==0.20.0" "accelerate==1.14.0" bitsandbytes sacrebleu


In [ ]:
# Load the tokenizer and inspect the generation prompt
import json
import torch
from transformers import AutoTokenizer
from google.colab import userdata

BASE_MODEL_ID = "NCAIR1/N-ATLaS"

with open(EVALUATION_SAMPLE_FILE, encoding="utf-8") as f:
    evaluation_records = [json.loads(line) for line in f if line.strip()]

# Retrieve the Hugging Face token from Colab Secrets
hf_token = userdata.get('hf_token')

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    token=hf_token,
)

example = evaluation_records[0]

prompt_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": example["input"]}],
    tokenize=False,
    add_generation_prompt=True,
)

print("English input:")
print(example["input"])

print("\nPrompt sent to the model:")
print(prompt_text)

print("\nReference Tiv target — for evaluation only:")
print(example["target"])

English input:
This advance knowledge brought with it responsibility .

Prompt sent to the model:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

This advance knowledge brought with it responsibility .<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Reference Tiv target — for evaluation only:
Er Yehova a pasen mbatomov nav kwagh u ken hemen yô , kwagh ne gema va ve a tom .


In [ ]:
# Load the 8-bit base model and attach the saved adapter


import os
import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from google.colab import userdata

assert torch.cuda.is_available(), "A GPU runtime is required."
assert (ADAPTER_DIR / "adapter_config.json").exists()

gc.collect()
torch.cuda.empty_cache()

# Retrieve the Hugging Face token from Colab Secrets
hf_token = userdata.get('hf_token')

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=hf_token,
    quantization_config=quantization_config,
    device_map={"": 0},
)

base_model.eval()

print("Base model loaded.")
print("8-bit:", getattr(base_model, "is_loaded_in_8bit", False))
print("GPU memory allocated (GiB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Base model loaded.
8-bit: True
GPU memory allocated (GiB): 8.47


In [ ]:
#Compare base and adapted generation

from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_DIR),
    is_trainable=False,
)
model.eval()

assert hasattr(model, "disable_adapter"), (
    "This PEFT model does not expose disable_adapter(). "
    "Stop here rather than compare against an invalid baseline."
)

def generate_translation(model, english_text, max_new_tokens=128):
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": english_text}],
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    # The model is loaded on GPU 0.
    inputs = {key: value.to("cuda:0") for key, value in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


predictions = []

for index, record in enumerate(evaluation_records, start=1):
    # Baseline: temporarily disable the adapter.
    with model.disable_adapter():
        base_prediction = generate_translation(model, record["input"])

    # Adapted model: normal adapter-enabled inference.
    adapted_prediction = generate_translation(model, record["input"])

    predictions.append({
        "id": record["id"],
        "input": record["input"],
        "reference": record["target"],
        "base_prediction": base_prediction,
        "adapted_prediction": adapted_prediction,
    })

    if index % 10 == 0:
        print(f"Completed {index}/{len(evaluation_records)} examples")
        gc.collect()

print("\nFirst comparison:")
print(json.dumps(predictions[0], ensure_ascii=False, indent=2))



Streaming output truncated to the last 5000 lines.


Completed 10/10 examples

First comparison:
{
  "id": "mt560-140929",
  "input": "This advance knowledge brought with it responsibility .",
  "reference": "Er Yehova a pasen mbatomov nav kwagh u ken hemen yô , kwagh ne gema va ve a tom .",
  "base_prediction": "What advance knowledge is being referred to?",
  "adapted_prediction": "What was the advance knowledge?"
}


In [ ]:
#Save predictions and calculate metrics

import json
from sacrebleu.metrics import BLEU, CHRF

PREDICTIONS_FILE = EVALUATION_DIR / "predictions.jsonl"
METRICS_FILE = EVALUATION_DIR / "metrics.json"

with open(PREDICTIONS_FILE, "w", encoding="utf-8") as f:
    for prediction in predictions:
        f.write(json.dumps(prediction, ensure_ascii=False) + "\n")

references = [item["reference"] for item in predictions]
base_outputs = [item["base_prediction"] for item in predictions]
adapted_outputs = [item["adapted_prediction"] for item in predictions]

bleu = BLEU()
chrf = CHRF()

def calculate_metrics(outputs, references):
    return {
        "exact_match_ratio": sum(
            pred.strip() == ref.strip()
            for pred, ref in zip(outputs, references)
        ) / len(references),
        "bleu": bleu.corpus_score(outputs, [references]).score,
        "chrf": chrf.corpus_score(outputs, [references]).score,
        "empty_predictions": sum(not pred.strip() for pred in outputs),
        "sample_count": len(references),
    }

metrics = {
    "evaluation_type": "validation_diagnostic",
    "dataset_split": "validation",
    "sample_size": len(predictions),
    "seed": 42,
    "base_model": calculate_metrics(base_outputs, references),
    "adapted_model": calculate_metrics(adapted_outputs, references),
}

with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("\nSaved predictions:", PREDICTIONS_FILE)
print("Saved metrics:", METRICS_FILE)


{
  "evaluation_type": "validation_diagnostic",
  "dataset_split": "validation",
  "sample_size": 10,
  "seed": 42,
  "base_model": {
    "exact_match_ratio": 0.0,
    "bleu": 0.5306470002591739,
    "chrf": 10.161518829075966,
    "empty_predictions": 0,
    "sample_count": 10
  },
  "adapted_model": {
    "exact_match_ratio": 0.0,
    "bleu": 0.5662523241748221,
    "chrf": 10.455352358298152,
    "empty_predictions": 0,
    "sample_count": 10
  }
}

Saved predictions: /content/drive/MyDrive/foundry/evaluations/tiv-lora-smoke-v1/predictions.jsonl
Saved metrics: /content/drive/MyDrive/foundry/evaluations/tiv-lora-smoke-v1/metrics.json


In [ ]:
# Release GPU memory

del model
del base_model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print("GPU memory allocated (GiB):",
      round(torch.cuda.memory_allocated() / 1024**3, 2))


GPU memory allocated (GiB): 0.01
